# Transfer functions, poles and zeros: notebook

The running example: the robot's heading loop (yaw inertia 0.1 kg·m², PD gains Kp = Kd = 0.4, RO-061's faster design, an integral gain Ki added) driving at 0.5 m/s while the worn right gearbox of RO-060 drags (a 0.02 N·m torque to the right), and its speed loop (mass 8 kg, drag 2 N·s/m) under PI control. Every number in the Note is computed here; the model lives in `images/tfkit.py`, shared with the figures.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import scipy.signal as sig
from tfkit import (M, B, I, KP, KD, D_DRAG, KI_GOOD, KI_BAD, KV, poles, drag_response, drive_path,
                   turn_with_delay, reverse_front, speed_pi)

## 1. The opening problem (Note, Section 1)

In [2]:
print('gearbox drag torque (negative: to the right):', D_DRAG, 'N m')
t = np.linspace(0, 30, 30001)
for ki in [0, KI_GOOD, KI_BAD]:
    th = drag_response(t, ki)
    x, y = drive_path(t, th)
    print(f'Ki = {ki}: heading at 30 s {th[-1]:.4f} rad, largest |heading| {np.abs(th).max():.3f} rad, '
          f'sideways at 30 s {y[-1]:.3f} m after {x[-1]:.2f} m')
th = drag_response(t, KI_BAD)
w = np.gradient(th, t)
ie = np.cumsum(-th) * (t[1] - t[0])
tq = KP * (-th) + KI_BAD * ie - KD * w
print('Ki = 2: torque stays between', tq.min().round(3), 'and', tq.max().round(3), 'N m')

gearbox drag torque (negative: to the right): -0.02 N m
Ki = 0: heading at 30 s -0.0500 rad, largest |heading| 0.050 rad, sideways at 30 s -0.725 m after 14.98 m
Ki = 0.2: heading at 30 s 0.0000 rad, largest |heading| 0.037 rad, sideways at 30 s -0.050 m after 15.00 m


Ki = 2.0: heading at 30 s -0.2481 rad, largest |heading| 0.302 rad, sideways at 30 s -0.049 m after 14.93 m
Ki = 2: torque stays between -0.107 and 0.165 N m


## 2. Transfer functions (Note, Section 2)

In [3]:
G = lambda z: 1 / (8 * z + 2)
print('G(-0.1) =', round(G(-0.1), 4), ' G(0.1) =', round(G(0.1), 4), ' G(0) =', G(0))
print('check 8 v\' + 2 v at s = -0.1:', 8 * (-0.1) * G(-0.1) + 2 * G(-0.1))
A = np.array([[0, 1], [0, 0]]); Bm = np.array([[0], [10]]); C = np.array([[1, 0]])
for z in [1.0, 2.0, 0.5]:
    inv = np.linalg.inv(z * np.eye(2) - A)
    print(f's = {z}: (sI - A)^-1 = {inv.tolist()}, formula [[1/s, 1/s^2], [0, 1/s]] = {[[1/z, 1/z**2], [0, 1/z]]}; C (sI - A)^-1 B = {(C @ inv @ Bm).item():.4f}, 10/s^2 = {10/z**2:.4f}')
print('characteristic polynomial of A (coefficients of s^2, s, 1):', np.poly(A), ' eigenvalues', np.linalg.eigvals(A))
Acl = np.array([[0, 1], [-4, -4]])
print('closed loop PD: characteristic polynomial', np.poly(Acl), ' eigenvalues', np.linalg.eigvals(Acl))

G(-0.1) = 0.8333  G(0.1) = 0.3571  G(0) = 0.5
check 8 v' + 2 v at s = -0.1: 1.0
s = 1.0: (sI - A)^-1 = [[1.0, 1.0], [0.0, 1.0]], formula [[1/s, 1/s^2], [0, 1/s]] = [[1.0, 1.0], [0, 1.0]]; C (sI - A)^-1 B = 10.0000, 10/s^2 = 10.0000
s = 2.0: (sI - A)^-1 = [[0.5, 0.25], [0.0, 0.5]], formula [[1/s, 1/s^2], [0, 1/s]] = [[0.5, 0.25], [0, 0.5]]; C (sI - A)^-1 B = 2.5000, 10/s^2 = 2.5000
s = 0.5: (sI - A)^-1 = [[2.0, 4.0], [0.0, 2.0]], formula [[1/s, 1/s^2], [0, 1/s]] = [[2.0, 4.0], [0, 2.0]]; C (sI - A)^-1 B = 40.0000, 10/s^2 = 40.0000
characteristic polynomial of A (coefficients of s^2, s, 1): [1. 0. 0.]  eigenvalues [0.+0.j 0.+0.j]
closed loop PD: characteristic polynomial [1. 4. 4.]  eigenvalues [-2.+0.j -2.+0.j]


## 3. Poles and zeros (Note, Section 3)

In [4]:
tt = np.linspace(0, 10, 100001)
_, y = sig.step(sig.lti([4], [1, 4, 4]), T=tt)
_, yz = sig.step(sig.lti([4, 4], [1, 4, 4]), T=tt)
y, yz = 0.5 * y, 0.5 * yz
print('D on measurement: largest heading', y.max().round(4))
print('D on error: peak', yz.max().round(4), 'rad at', tt[yz.argmax()].round(3), 's; overshoot', ((yz.max() - 0.5) / 0.5).round(4), '; e^-2 =', np.exp(-2).round(4))
print('check yz = y + y\':', np.abs(yz - (y + np.gradient(y, tt))).max().round(5))
print('formula 0.5 - 0.5 e^-2t + t e^-2t at t = 1:', round(0.5 - 0.5 * np.exp(-2) + np.exp(-2), 4))

D on measurement: largest heading 0.5
D on error: peak 0.5677 rad at 1.0 s; overshoot 0.1353 ; e^-2 = 0.1353
check yz = y + y': 0.0001
formula 0.5 - 0.5 e^-2t + t e^-2t at t = 1: 0.5677


## 4. BIBO stability (Note, Section 4)

In [5]:
t = np.linspace(0, 20, 20001)
print('speed plant, 1 N: speed at 20 s', round(0.5 * (1 - np.exp(-5)), 4), '-> 0.5 m/s; 4 N -> at most', 4 / B, 'm/s')
print('heading plant, 0.02 N m: theta = 0.02 / (2 * 0.1) t^2 = 0.1 t^2 ->', 0.1 * 20**2, 'rad at 20 s')
w = 0.02 * np.sin(2 * t)
_, thP, _ = sig.lsim(sig.lti([10], [1, 0, 4]), U=w, T=t)
_, thPD, _ = sig.lsim(sig.lti([10], [1, 4, 4]), U=w, T=t)
print('P only: largest heading by 20 s', np.abs(thP).max().round(3), '; growth 0.05 rad per s, 0.05 * 20 =', 0.05 * 20)
print('PD: largest heading', np.abs(thPD).max().round(4))

speed plant, 1 N: speed at 20 s 0.4966 -> 0.5 m/s; 4 N -> at most 2.0 m/s
heading plant, 0.02 N m: theta = 0.02 / (2 * 0.1) t^2 = 0.1 t^2 -> 40.0 rad at 20 s
P only: largest heading by 20 s 0.942 ; growth 0.05 rad per s, 0.05 * 20 = 1.0
PD: largest heading 0.0297


## 5. Block-diagram algebra (Note, Section 5)

In [6]:
ki = 0.2
for z in [1.0, 2.0, 0.5 + 1j]:
    C_ = 0.4 + ki / z; P_ = 10 / z**2; H_ = 0.4 * z
    inner = P_ / (1 + H_ * P_); fwd = C_ * inner; T_ = fwd / (1 + fwd); Gd = inner / (1 + fwd)
    print(f's = {z}: parallel {C_:.4f} vs (0.4 s + Ki)/s {(0.4 * z + ki) / z:.4f}; inner {inner:.4f} vs 10/(s^2 + 4 s) {10 / (z**2 + 4 * z):.4f}')
    print(f'      series {fwd:.4f} vs (4 s + 10 Ki)/(s^3 + 4 s^2) {(4 * z + 10 * ki) / (z**3 + 4 * z**2):.4f}; closed loop {T_:.4f} vs {(4 * z + 10 * ki) / (z**3 + 4 * z**2 + 4 * z + 10 * ki):.4f}')
    print(f'      drag torque to heading {Gd:.4f} vs 10 s/(s^3 + 4 s^2 + 4 s + 10 Ki) {10 * z / (z**3 + 4 * z**2 + 4 * z + 10 * ki):.4f}')

s = 1.0: parallel 0.6000 vs (0.4 s + Ki)/s 0.6000; inner 2.0000 vs 10/(s^2 + 4 s) 2.0000
      series 1.2000 vs (4 s + 10 Ki)/(s^3 + 4 s^2) 1.2000; closed loop 0.5455 vs 0.5455
      drag torque to heading 0.9091 vs 10 s/(s^3 + 4 s^2 + 4 s + 10 Ki) 0.9091
s = 2.0: parallel 0.5000 vs (0.4 s + Ki)/s 0.5000; inner 0.8333 vs 10/(s^2 + 4 s) 0.8333
      series 0.4167 vs (4 s + 10 Ki)/(s^3 + 4 s^2) 0.4167; closed loop 0.2941 vs 0.2941
      drag torque to heading 0.5882 vs 10 s/(s^3 + 4 s^2 + 4 s + 10 Ki) 0.5882
s = (0.5+1j): parallel 0.4800-0.1600j vs (0.4 s + Ki)/s 0.4800-0.1600j; inner 0.4706-1.8824j vs 10/(s^2 + 4 s) 0.4706-1.8824j
      series -0.0753-0.9788j vs (4 s + 10 Ki)/(s^3 + 4 s^2) -0.0753-0.9788j; closed loop 0.4900-0.5398j vs 0.4900-0.5398j
      drag torque to heading 1.2562-0.7059j vs 10 s/(s^3 + 4 s^2 + 4 s + 10 Ki) 1.2562-0.7059j


## 6. Root locus and Routh-Hurwitz (Note, Section 6)

In [7]:
for ki in [0, 0.1185, 0.2, 0.4, 1.0, 1.6, 2.0]:
    print(f'Ki = {ki}: poles {np.round(np.sort_complex(poles(ki)), 3)}')
ks = np.linspace(0.001, 0.3, 30000)
gap = [np.min(np.abs(np.diff(np.sort(poles(k).real)))) + np.abs(poles(k).imag).max() for k in ks]
print('the two slow poles meet near Ki =', ks[int(np.argmin(gap))].round(4))
t = np.linspace(0, 30, 30001)
th = drag_response(t, 0.2)
print('Ki = 0.2: largest swing', np.abs(th).max().round(4), 'rad (to the right) at', t[np.abs(th).argmax()].round(2), 's, within 0.005 rad from', t[np.where(np.abs(th) > 0.005)[0][-1]], 's')
th12 = drag_response(t, 0.1185)
print('Ki = 0.1185: within 0.005 rad from', t[np.where(np.abs(th12) > 0.005)[0][-1]], 's')
print('Routh-Hurwitz for s^3 + 4 s^2 + 4 s + 10 Ki: 4 x 4 > 10 Ki -> Ki <', 4 * 4 / 10)
print('RO-060 loop (Kp 0.2, Kd 0.283): 2.83 x 2 > 10 Ki -> Ki <', round(2.83 * 2 / 10, 3), '; poles at Ki = 0.6:', np.round(np.roots([0.1, 0.283, 0.2, 0.6]), 3))
z = 2j
print('check s = 2j at Ki = 1.6:', z**3 + 4 * z**2 + 4 * z + 16)

Ki = 0: poles [-2.+0.j -2.+0.j  0.+0.j]
Ki = 0.1185: poles [-2.667+0.j -0.676+0.j -0.657+0.j]
Ki = 0.2: poles [-2.839+0.j    -0.58 -0.606j -0.58 +0.606j]
Ki = 0.4: poles [-3.13 +0.j    -0.435-1.043j -0.435+1.043j]
Ki = 1.0: poles [-3.654+0.j    -0.173-1.645j -0.173+1.645j]
Ki = 1.6: poles [-4.+0.j  0.-2.j  0.+2.j]
Ki = 2.0: poles [-4.186+0.j     0.093-2.184j  0.093+2.184j]


the two slow poles meet near Ki = 0.1185


Ki = 0.2: largest swing 0.0369 rad (to the right) at 1.74 s, within 0.005 rad from 4.68 s
Ki = 0.1185: within 0.005 rad from 7.389 s
Routh-Hurwitz for s^3 + 4 s^2 + 4 s + 10 Ki: 4 x 4 > 10 Ki -> Ki < 1.6
RO-060 loop (Kp 0.2, Kd 0.283): 2.83 x 2 > 10 Ki -> Ki < 0.566 ; poles at Ki = 0.6: [-2.863+0.j     0.017+1.447j  0.017-1.447j]
check s = 2j at Ki = 1.6: 0j


## 7. Pole-zero cancellation (Note, Section 7)

In [8]:
t = np.linspace(0, 25, 25001)
for ki in [5.0, 10.0]:
    v = speed_pi(t, ki)
    pre = np.where((t < 10) & (np.abs(v - 0.5) > 0.01))[0]
    post = np.where((t >= 10) & (np.abs(v - 0.5) > 0.01))[0]
    after = v[t >= 10]
    print(f'Ki = {ki}: zero at {-ki / KV}, poles {np.round(np.roots([M, B + KV, ki]), 3)}, largest speed before the ramp {v[t < 10].max():.4f}, within 0.01 from {t[pre[-1] + 1]:.3f} s;'
          f' lowest speed {after.min():.4f} at {t[t >= 10][after.argmin()] - 10:.2f} s after the ramp starts, back within 0.01 after {t[post[-1] + 1] - 10:.2f} s')
tp = np.log(10) / 2.25
print('0.785 / 18 =', round(0.785 / 18, 4), '; deepest at ln 10 / 2.25 =', round(tp, 3), 's:', round(-0.785 / 18 * (np.exp(-0.25 * tp) - np.exp(-2.5 * tp)), 4))
print('values: e^-0.25t =', round(np.exp(-0.25 * tp), 4), ' e^-2.5t =', round(np.exp(-2.5 * tp), 4), ' product', round(0.0436 * (0.7743 - 0.0774), 4))
print('reference time constant 1 / 2.5 =', 1 / 2.5, 's')

Ki = 5.0: zero at -0.25, poles [-2.5  -0.25], largest speed before the ramp 0.5000, within 0.01 from 0.922 s; lowest speed 0.4696 at 1.02 s after the ramp starts, back within 0.01 after 5.89 s
Ki = 10.0: zero at -0.5, poles [-2.175 -0.575], largest speed before the ramp 0.5049, within 0.01 from 0.759 s; lowest speed 0.4721 at 0.83 s after the ramp starts, back within 0.01 after 3.14 s
0.785 / 18 = 0.0436 ; deepest at ln 10 / 2.25 = 1.023 s: -0.0304
values: e^-0.25t = 0.7743  e^-2.5t = 0.0774  product 0.0304
reference time constant 1 / 2.5 = 0.4 s


## 8. Non-minimum phase (Note, Section 8)

In [9]:
t = np.linspace(0, 5, 50001)
for v in [0.1, -0.1]:
    x, y, th, yf = reverse_front(t, v, a=0.1)
    zc = t[np.where(yf[1:] * yf[:-1] < 0)[0] + 1]
    print(f'v = {v}: zero at s = {-v / 0.1:g}; front point: largest {yf.max():.4f} m at {t[yf.argmax()]:.3f} s, at 5 s {yf[-1]:.4f} m, crosses zero at {zc}')

v = 0.1: zero at s = -1; front point: largest 0.2409 m at 5.000 s, at 5 s 0.2409 m, crosses zero at []
v = -0.1: zero at s = 1; front point: largest 0.0159 m at 0.938 s, at 5 s -0.1451 m, crosses zero at [1.7807]


## 9. Time delay (Note, Section 9)

In [10]:
t = np.linspace(0, 15, 15001)
for d in [0, 0.01, 0.17, 0.3, 0.33]:
    th = turn_with_delay(t, d)
    out = np.where(np.abs(th - 0.5) > 0.01)[0]
    st = t[out[-1] + 1] if out[-1] + 1 < len(t) else None
    print(f'delay {d} s: peak {th.max():.4f} rad, overshoot {100 * max(0, th.max() - 0.5) / 0.5:.1f} %, settled {st} s')
from scipy.optimize import brentq
w = np.sqrt(8 + np.sqrt(80))
print('largest delay (the RO-064 way): |0.4 + 0.4 j w| = 0.1 w^2 at w =', round(w, 3), 'rad/s, delay = atan(w) / w =', round(np.arctan(w) / w, 3), 's')

delay 0 s: peak 0.5000 rad, overshoot 0.0 %, settled 2.92 s
delay 0.01 s: peak 0.5000 rad, overshoot 0.0 %, settled 2.918 s
delay 0.17 s: peak 0.5000 rad, overshoot 0.0 %, settled 2.927 s


delay 0.3 s: peak 0.5729 rad, overshoot 14.6 %, settled 13.403 s
delay 0.33 s: peak 0.8357 rad, overshoot 67.1 %, settled None s
largest delay (the RO-064 way): |0.4 + 0.4 j w| = 0.1 w^2 at w = 4.116 rad/s, delay = atan(w) / w = 0.324 s


## 10. SISO and MIMO (Note, Section 10)

In [11]:
u, tau = 1.0, 0.1
FR, FL = u / 2 + 5 * tau, u / 2 - 5 * tau
print('F_R =', FR, ' F_L =', FL, ' sum', FR + FL, ' torque (F_R - F_L) x 0.1 =', (FR - FL) * 0.1)
print('heading from wheel forces: torque (F_R - F_L) x 0.1 m into 1/(0.1 s^2) gives (F_R - F_L)/s^2')

F_R = 1.0  F_L = 0.0  sum 1.0  torque (F_R - F_L) x 0.1 = 0.1
heading from wheel forces: torque (F_R - F_L) x 0.1 m into 1/(0.1 s^2) gives (F_R - F_L)/s^2
